# FPN — Feature Pyramid Networks for Object Detection (2016)

_Papers · Computer Vision_

**Add a top-down pathway with lateral connections to a plain backbone, turning its one-way feature hierarchy into a multi-scale pyramid that is high-resolution AND semantically strong at every level.**

---

This notebook is a paced, step-by-step walkthrough of the **FPN — Feature Pyramid Networks for Object Detection (2016)** lesson. We build the merge one piece at a time: a single hand-checked merge, the full top-down/lateral module, and an ablation that shows what the lateral connection buys us. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build FPN in four steps: (1) a single merge by hand so the arithmetic is concrete, (2) the full top-down/lateral module over a toy backbone, (3) plant a sharp feature for an ablation, (4) measure how much of that feature survives with vs without the lateral connection.

### Step 1 — One merge by hand: combine C4 with M5 into M4

Each FPN level is built by merging two maps: a **lateral** $1\times1$ conv of the bottom-up feature $C_4$ (which fixes the channel count to $d=256$ but keeps resolution), and the **upsampled** coarser merged map $M_5$ (nearest-neighbor, $2\times$). The two are added element-wise.

We set one cell of each to known values (lateral $=0.30$, upsampled $=0.50$) so we can check the add gives $0.80$. A final $3\times3$ conv smooths the upsampling aliasing into $P_4$ without changing the shape.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

D = 256  # FPN channel width d (paper sets d=256)

# Worked example: merge C4 (512,4,4) with M5 (256,2,2) -> M4 (256,4,4).
C4 = torch.zeros(1, 512, 4, 4)      # bottom-up, stride 16, 512 channels
M5 = torch.zeros(1, 256, 2, 2)      # coarser merged map, stride 32, 256 channels

lateral4 = nn.Conv2d(512, D, kernel_size=1)   # 1x1: 512 -> 256, keeps 4x4
with torch.no_grad():                          # zero it so we can check 0.30 + 0.50 = 0.80
    lateral4.weight.zero_()
    lateral4.bias.zero_()

lat = lateral4(C4)                             # (1,256,4,4)
lat[0, 0, 0, 0] = 0.30                         # lateral value at ch0,row0,col0

up = F.interpolate(M5, scale_factor=2, mode="nearest")  # (1,256,2,2) -> (1,256,4,4)
up[0, 0, 0, 0] = 0.50                          # upsampled top-down value at same cell

M4 = lat + up                                  # element-wise add -> (1,256,4,4)
print("shapes:  lateral(C4)=", tuple(lat.shape[1:]),
      " up(M5)=", tuple(up.shape[1:]), " M4=", tuple(M4.shape[1:]))
print("worked cell  0.30 + 0.50 =", round(M4[0, 0, 0, 0].item(), 4), "(expect 0.80)")

smooth4 = nn.Conv2d(D, D, kernel_size=3, padding=1)    # 3x3 anti-alias -> P4, shape unchanged
P4 = smooth4(M4)
print("P4 shape (unchanged by 3x3):", tuple(P4.shape[1:]))

### Step 2 — The full FPN module over a toy backbone

Now we generalize that one merge into the whole pyramid. The `FPN` module starts at the coarsest map $C_5$, then walks down the levels: at each step it upsamples the running merged map and adds the lateral $1\times1$ projection of the next bottom-up map. A per-level $3\times3$ conv produces the final $P_2\dots P_5$.

We run it on a toy backbone whose maps have strides $\{4,8,16,32\}$ on a $64\times64$ image. Notice every output level ends up with $256$ channels — high resolution AND a uniform semantic width.

In [ ]:
class FPN(nn.Module):
    def __init__(self, in_channels, d=256):          # in_channels for [C2,C3,C4,C5]
        super().__init__()
        self.lateral = nn.ModuleList([nn.Conv2d(c, d, 1) for c in in_channels])
        self.smooth  = nn.ModuleList([nn.Conv2d(d, d, 3, padding=1) for _ in in_channels])

    def forward(self, cs):                            # cs = [C2,C3,C4,C5], coarsest last
        M = self.lateral[-1](cs[-1])                  # M5 = 1x1(C5): start the iteration on C5
        Ms = [M]
        for i in range(len(cs) - 2, -1, -1):          # walk down C4, C3, C2
            up = F.interpolate(M, scale_factor=2, mode="nearest")
            M = self.lateral[i](cs[i]) + up           # 1x1 lateral + upsampled top-down, ADDED
            Ms.insert(0, M)
        return [self.smooth[i](Ms[i]) for i in range(len(cs))]   # P2..P5 = 3x3(M)

# Toy backbone maps on a 64x64 "image": strides {4,8,16,32} -> sizes {16,8,4,2}.
C = [torch.randn(1, 256, 16, 16),
     torch.randn(1, 512, 8, 8),
     torch.randn(1, 1024, 4, 4),
     torch.randn(1, 2048, 2, 2)]   # C2,C3,C4,C5

fpn = FPN(in_channels=[256, 512, 1024, 2048], d=D)
Ps = fpn(C)

print("Pyramid levels (each is high-res AND semantically strong):")
for name, c, p in zip(["P2", "P3", "P4", "P5"], C, Ps):
    print(f"  {name}: resolution {tuple(p.shape[2:])}, channels {p.shape[1]}  "
          f"(from C of res {tuple(c.shape[2:])}, channels {c.shape[1]})")
# Every P level has 256 channels; resolutions match {16,8,4,2} -> all scales covered.

### Step 3 — Plant a sharp feature for the ablation

To show what the lateral connection contributes, we set up a controlled test. The high-resolution bottom-up map $C_2$ gets a single **sharp localized spike** at position $(4,11)$ — a precise feature the coarse $C_5$ cannot carry. $C_5$ is just small random noise standing in for coarse semantics with no fine location.

We then form two candidate merged maps: `full_fpn` adds the lateral $C_2$ back in, while `no_lat` is the top-down map only (the lateral connection dropped). Everything else is identical.

In [ ]:
# ABLATION: full merge vs top-down-only (drop the lateral add).
# Plant a sharp localized spike in the high-res bottom-up map C2 that the coarse C5 cannot carry.
torch.manual_seed(1)
C2 = torch.zeros(1, D, 16, 16)
C2[0, :, 4, 11] = 5.0                                    # a sharp localized feature at (4,11)
C5 = torch.randn(1, D, 2, 2) * 0.1                       # coarse, semantic, no fine location

def up_to(x, size):
    return F.interpolate(x, size=size, mode="nearest")

top_down = up_to(C5, (16, 16))                           # coarse map blown up to 16x16 (blurry)
full_fpn = C2 + top_down                                 # WITH lateral add
no_lat   = top_down                                      # top-down ONLY (lateral dropped)

### Step 4 — Measure how much of the sharp feature survives

We quantify localization with a simple contrast: the magnitude at the spike's location $(4,11)$ relative to the map's average magnitude. A high number means the sharp feature is still distinct; a low one means it washed out.

The full FPN should keep the spike sharp, while the top-down-only map smears it away — demonstrating that the lateral connection is what carries fine localization into the pyramid.

In [ ]:
# How much of the sharp spike at (4,11) survives, relative to the map's mean magnitude?
def peak_contrast(x):
    peak = x[0, :, 4, 11].abs().mean()
    background = x.abs().mean() + 1e-9
    return (peak / background).item()

print("Ablation -- sharp-feature contrast at (4,11) (higher = localization preserved):")
print(f"  full FPN (with lateral):    {peak_contrast(full_fpn):.2f}")
print(f"  top-down only (no lateral): {peak_contrast(no_lat):.2f}")
# full FPN keeps the spike sharp; top-down-only washes it out -> lateral connections carry location.
# (Our small run, not the paper's reported number.)

## Visualize the data & results

_Does the lateral connection preserve fine localization that a top-down-only pyramid washes out?_

### Step 1 — Re-plant the spike and form both merged maps

This panel re-runs the ablation in a self-contained way so we can read off actual numbers along a row. We plant the same sharp spike at $(4,11)$ in the high-res map $C_2$, and make the coarse $C_5$ a smooth constant with no fine location.

We then build `full_fpn` (with the lateral add) and `no_lat` (top-down only) exactly as before.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(1)

D = 256
# Plant a sharp localized feature in the high-res bottom-up map; coarse top-down map has only semantics.
C2 = torch.zeros(1, D, 16, 16)
C2[0, :, 4, 11] = 5.0                                      # sharp spike at row 4, col 11
C5 = torch.zeros(1, D, 2, 2) + 0.1                         # coarse, smooth, no fine location

top_down = F.interpolate(C5, size=(16, 16), mode="nearest")  # blow coarse map up to 16x16 (blurry)
full_fpn = C2 + top_down                                   # WITH lateral add
no_lat   = top_down                                        # top-down ONLY

### Step 2 — Read magnitudes along the spike's row

We slice out row 4 (columns 6 through 15) of channel 0 from each map and print the magnitudes side by side. The full-FPN row should show a clear spike of about $5.1$ at column 11; the top-down-only row should be flat near $0.1$ — the spike is simply gone.

That contrast is the whole point: only the lateral connection carries the precise location into the merged map.

In [ ]:
row_full  = full_fpn[0, 0, 4, 6:16]   # magnitudes along row 4, cols 6..15
row_nolat = no_lat[0, 0, 4, 6:16]

print("col      :", list(range(6, 16)))
print("full FPN :", [round(v.item(), 2) for v in row_full])   # spike ~5.1 at col 11
print("top-down :", [round(v.item(), 2) for v in row_nolat])  # flat ~0.1 -> spike lost
# full FPN preserves the localized spike; top-down-only washes it out -> the lateral connection
# is what carries fine localization. (Our small run, not the paper's reported number.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Take the full FPN merge (upsample top-down + $1\times1$ lateral, then add)
            and compare it against a top-down-only variant that drops the lateral add (it returns just
            the upsampled coarse map). Feed both a toy bottom-up map that carries a sharp, localized feature
            (a single bright spot) the coarse map has lost. How well does each output preserve that sharp
            feature, and what does the gap demonstrate about why FPN needs lateral connections?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Keep everything else identical &mdash; same inputs, same upsample, same shapes &mdash; and change only whether the bottom-up lateral map is added back. — _An honest ablation changes exactly one thing &mdash; the lateral connection &mdash; so any difference in the output is attributable to it._
- Measure how much of the bottom-up map's sharp localized signal survives in each output (e.g. correlation of the output with the true sharp feature, or recovered peak sharpness). — _The lateral term is the ONLY path that carries the high-resolution backbone map's location into the pyramid; the upsampled top-down term is blurry by construction._
- Observe that full-FPN recovers the sharp feature while top-down-only outputs a smeared blob with the localized detail missing. — _Nearest-neighbor upsampling copies coarse pixels; without the lateral add there is no fine-resolution source, so small/precise structure cannot appear._

**Answer:** Full FPN preserves the sharp localized feature (high correlation with the true signal)
                 because the lateral connection injects the high-resolution bottom-up map. The
                 top-down-only variant outputs a blurred version &mdash; the localized detail is gone &mdash;
                 because its only source is a $2\times$-upsampled coarse map, which carries semantics but no
                 fine location. Since only the lateral add changed, this isolates the lateral
                 connection as the source of precise localization, matching the paper's Table 1 finding
                 that removing it sharply lowers recall, hardest on small objects.

</details>

**Problem 2.** Trace the shapes through one merge at level $\ell = 3$. The bottom-up map $C_3$ has stride 8,
            shape $(256_{\text{in}}, 8, 8)$ on a $64\times64$ toy image, and the coarser merged map $M_4$
            has shape $(256, 4, 4)$. Give the shape after the $1\times1$ lateral conv, after upsampling
            $M_4$, after the add, and after the $3\times3$ conv (call the result $P_3$).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $1\times1$ conv on $C_3$ maps channels to $256$ and keeps spatial size: $(256_{\text{in}},8,8) \to (256,8,8)$. — _A $1\times1$ conv is a per-pixel channel projection; it never changes height or width._
- Upsample $M_4$ by $2\times$ nearest-neighbor: $(256,4,4) \to (256,8,8)$. — _Doubling height and width brings the coarse top-down map up to $C_3$'s stride-8 resolution so the add is shape-legal._
- Element-wise add the two $(256,8,8)$ maps $\to (256,8,8)$; then $3\times3$ same-padding conv $\to (256,8,8) = P_3$. — _Addition needs equal shapes (now satisfied); the $3\times3$ conv smooths the upsampling aliasing without changing the shape._

**Answer:** After lateral $1\times1$: $(256,8,8)$. After upsampling $M_4$: $(256,8,8)$. After the add:
                 $(256,8,8)$. After the $3\times3$ conv: $(256,8,8) = P_3$. Every step keeps the
                 $8\times8$ resolution and $256$ channels once the upsample and $1\times1$ have lined the
                 operands up &mdash; exactly the shape bookkeeping the notebook prints.

</details>

**Problem 3.** In the worked example, a single position had $\text{lateral}(C_4) = 0.30$ and upsampled
            $\text{up}(M_5) = 0.50$, giving merged $M_4 = 0.80$ there. Now suppose at a neighboring position
            the lateral term is a sharp localized spike of $0.90$ while the upsampled top-down term is only
            $0.10$ (it blurred the spike away). What is the merged value, and what would the
            top-down-only ablation output at that same position instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Full FPN adds the two terms: $M_4 = 0.90 + 0.10 = 1.00$ at that position. — _The element-wise add fuses both signals; the sharp lateral spike dominates because the top-down term blurred it to $0.10$._
- Top-down-only drops the lateral add and returns just the upsampled term: $0.10$ at that position. — _Without the lateral connection there is no high-resolution source, so the sharp spike never enters the output._
- Compare: $1.00$ (sharp peak preserved) vs $0.10$ (peak lost). — _The gap of $0.90$ is exactly the localized signal that only the lateral connection carries._

**Answer:** Full FPN outputs $0.90 + 0.10 = \mathbf{1.00}$, preserving the sharp spike; the
                 top-down-only ablation outputs just $\mathbf{0.10}$, losing it. The missing $0.90$ is the
                 fine localization signal the lateral connection supplies &mdash; a single-position version of
                 why removing lateral connections hurts recall in the paper's Table 1.

</details>